In [ ]:
import gffutils
import pyfaidx
from Bio.Seq import Seq
import subprocess
from Bio import SeqIO

from Bio import Phylo



In [ ]:

# db = gffutils.create_db(gfff, dbfn=':memory:')

# geneids = []
# for cds in db.features_of_type('CDS', order_by='start'):
#     gene = cds.attributes['gene'][0]
#     geneids.append(gene)

# geneids

In [ ]:
def extract_gene_sequences(fasta_path, gff_path, gene_name, nuc_out, aa_out):
    """
    Extract nucleotide and amino acid sequences for a specific gene from a multi-sequence FASTA and GFF3.
    Args:
        fasta_path (str): Path to input FASTA file (aligned or unaligned).
        gff_path (str): Path to GFF3 file.
        gene_name (str): Name of the gene to extract.
        nuc_out (str): Output path for nucleotide sequences.
        aa_out (str): Output path for amino acid sequences.
    """
    db = gffutils.create_db(gff_path, dbfn=':memory:')
    fasta = pyfaidx.Fasta(fasta_path)
    # Find the CDS feature for the gene
    cds_list = [cds for cds in db.features_of_type('CDS', order_by='start') if cds.attributes.get('gene', [''])[0] == gene_name]
    if not cds_list:
        raise ValueError(f"Gene {gene_name} not found in GFF3.")
    cds = cds_list[0]
    seqid = cds.seqid
    start = cds.start - 1  # pyfaidx uses 0-based indexing
    end = cds.end

    # Clear output files
    open(aa_out, "w").close()
    open(nuc_out, "w").close()

    with open(aa_out, "a") as faa, open(nuc_out, "a") as fnuc:
        for seq_record in list(fasta.keys()):
            nstops = [Seq(fasta[seq_record][(start+n):(end+n)].seq.replace('-', 'N')).translate().count('*') for n in [0,1,2]]
            n = nstops.index(min(nstops))
            nseq = fasta[seq_record][(start+n):(end+n)].seq
            aa = Seq(nseq.replace('-', 'N')).translate()
            faa.write(f">{seq_record}    {gene_name} (n={n})\n{aa}\n")
            fnuc.write(f">{seq_record}    {gene_name}\n{nseq.replace('-', 'N')}\n")



def filter_n_sequences(nuc_fasta, aa_fasta, nuc_out, aa_out, n_threshold=0.2):
    """
    Remove sequences from nucleotide and amino acid FASTA files if the nucleotide sequence has > n_threshold fraction of 'n'.
    Args:
        nuc_fasta (str): Path to nucleotide FASTA file.
        aa_fasta (str): Path to amino acid FASTA file.
        nuc_out (str): Output path for filtered nucleotide FASTA.
        aa_out (str): Output path for filtered amino acid FASTA.
        n_threshold (float): Fraction threshold for 'n' bases (default 0.2).
    """
    nuc_records = SeqIO.to_dict(SeqIO.parse(nuc_fasta, "fasta"))
    aa_records = SeqIO.to_dict(SeqIO.parse(aa_fasta, "fasta"))

    keep_ids = []
    drop_ids = []
    for rec_id, rec in nuc_records.items():
        seq = str(rec.seq).lower()
        n_frac = seq.count('n') / len(seq) if len(seq) > 0 else 1.0
        if n_frac <= n_threshold:
            keep_ids.append(rec_id)
        else:
            drop_ids.append(rec_id)

    with open(nuc_out, "w") as nf, open(aa_out, "w") as af:
        SeqIO.write([nuc_records[i] for i in keep_ids if i in nuc_records], nf, "fasta")
        SeqIO.write([aa_records[i] for i in keep_ids if i in aa_records], af, "fasta")
    return(len(keep_ids), len(drop_ids))

In [ ]:
def extract_reference_gene_aa(fasta_path, gff_path, prefix=""):
    """
    Extract reference amino acid sequences for each CDS from a reference fasta and gff,
    and write all to a single output file.
    Args:
        fasta_path (str): Path to reference FASTA file.
        gff_path (str): Path to GFF3 file.
        output_dir (str): Output directory for amino acid sequences.
    """
    ref_fasta = pyfaidx.Fasta(fasta_path)
    db = gffutils.create_db(gff_path, dbfn=':memory:')
    for cds in db.features_of_type('CDS', order_by='start'):
        gene = cds.attributes['gene'][0]
        with open(f"{prefix}{gene}_aa.fasta", "w") as f:
            seqid = cds.seqid
            start = cds.start - 1  # pyfaidx uses 0-based
            end = cds.end
            ref_seq = ref_fasta[seqid][start:end]
            nstops = [Seq(ref_seq[(n):(end-start+n)].seq.replace('-', 'N')).translate().count('*') for n in [0,1,2]]
            n = nstops.index(min(nstops))
            nseq = ref_seq[(n):(end-start+n)].seq
            aa = Seq(nseq.replace('-', 'N')).translate()
            f.write(f">{seqid}    {gene} (n={n})\n{aa}\n")


In [ ]:
# # get sequences for gene region for all consensus sequences
# # filter sequences with > 20% Ns

# for cds in db.features_of_type('CDS', order_by='start'):
#     gene = cds.attributes['gene'][0]
#     extract_gene_sequences(fastaf, gfff, gene, f"geneseqs/{gene}_sequences.fasta", f"geneseqs/{target}_{gene}_aa.fasta")

#     nkeep, ndrop = filter_n_sequences(f"geneseqs/{gene}_sequences.fasta", f"geneseqs/{target}_{gene}_aa.fasta", 
#                        f"geneseqs/{gene}_sequences_filt.fasta", f"geneseqs/{target}_{gene}_aa_filt.fasta", 
#                         n_threshold=0.2)
#     print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")


In [7]:
def run_iqtree(iqprefix, codonf):
    """Run IQ-TREE on the codon alignment.""" 
    subprocess.run(["iqtree", "--prefix", iqprefix, "-s", codonf, 
                "-m", "GTR+G", "-nt", "AUTO", "-bb", "1000", "-alrt", "1000",
                "--redo"], check=True)


In [8]:
def get_codon_alignment(gaafastaf, gfastaf, reffasta, gmafftf, codonf):
    """Generate a codon alignment using pal2nal from an amino acid alignment and the corresponding nucleotide sequences."""
    pal2nal = "/opt/miniconda3/envs/hyphy/bin/pal2nal.pl"

    #run mafft to align aa sequences
    command = ["mafft", '--keeplength',
               #'--localpair', '--maxiterate', '1000', '--op', '3.0',
               '--auto', '--op', '3.0',
               '--anysymbol', '--add', gaafastaf, reffasta]
    print(" ".join(command))
    with open(gmafftf, 'w') as outfile, open(gmafftf + ".err", 'w') as errfile:
        subprocess.run(command, stdout=outfile, stderr=errfile, check=True)

    # Parse MAFFT output and remove the original reference sequence from reffasta
    ref_id = list(SeqIO.parse(reffasta, "fasta"))[0].id
    aligned_records = list(SeqIO.parse(gmafftf, "fasta"))
    filtered_records = [rec for rec in aligned_records if rec.id != ref_id]
    # Overwrite gmafftf with filtered records (without the original reference)
    with open(gmafftf, "w") as out_handle:
        SeqIO.write(filtered_records, out_handle, "fasta")


    # Run pal2nal to generate codon alignment from the MAFFT output and the nucleotide fasta
    command = [pal2nal, gmafftf, gfastaf, "-nomismatch", "-nogap", "-output", "fasta"]
    print(" ".join(command))
    with open(codonf, "w") as outfile:
        subprocess.run(command, stdout=outfile, check=True)

def trim_stop_codons(codonf, codontrimf, mincodons=10):
    """Trim codon alignment to remove sequences after the first stop codon found in any sequence."""
    # Define stop codons
    STOP_CODONS = {"TAA", "TAG", "TGA"}

    records = list(SeqIO.parse(codonf, "fasta"))
    L = len(records[0].seq)

    n_codons = L // 3
    trim_at = n_codons  # default: keep full length
    checkcodons = min(mincodons, n_codons)  # check up to last 'mincodons'(10) codons

    # Look for stops in the last 'mincodons' codons
    for rec in records:
        codons = [str(rec.seq[i:i+3]).upper() for i in range(0, L, 3)]
        for offset in range(checkcodons, 0, -1):  # check from -checkcodons to -1 codons
            idx = n_codons - offset
            if idx < 0:
                continue
            if codons[idx] in STOP_CODONS:
                trim_at = min(trim_at, idx)  # earliest stop position
                break

    # Convert codon index back to nt length
    trim_nt = trim_at * 3
    print(f"Trimming alignment {L//3} -> {trim_at} codons ({trim_nt} nt).")

    with open(codontrimf, "w") as out:
        for rec in records:
            rec.seq = rec.seq[:trim_nt]
            SeqIO.write(rec, out, "fasta")


In [9]:
def remove_zero_length_branches(tree):
    """
    Removes clades with a branch_length of 0 from a Biopython Tree object.
    This effectively prunes zero-length branches and creates polytomies.
    """
    # Create a list to store clades to be processed (children of zero-length clades)
    clades_to_process = []

    # Iterate through all clades in the tree
    for clade in tree.find_clades():
        # Check if the clade has children and a branch_length of 0
        if clade.branch_length == 0 and clade.clades:
            clades_to_process.append(clade)
    print(f"Removing {len(clades_to_process)} zero-length branches")
    # Process clades with zero-length branches
    for zero_clade in clades_to_process:
        parent = tree.get_path(zero_clade)[-2]  # Get the parent of the zero-length clade
        
        if parent:
            # Remove the zero-length clade from its parent's children
            parent.clades.remove(zero_clade)
            # Add the children of the zero-length clade directly to the parent
            parent.clades.extend(zero_clade.clades)

    return tree

def prune_tree_to_seqs(wgstree, fastaf, pruned_tree_file):
    """
    Prune a tree to only include tips present in the fasta file.
    Args:
        wgstree (str): Path to the input tree file (Newick format).
        fastaf (str): Path to the fasta file containing sequence names to keep.
        pruned_tree_file (str): Path to write the pruned tree.
    """
    seq_names = set(SeqIO.to_dict(SeqIO.parse(fastaf, "fasta")).keys())
    
    # Read the tree
    tree = Phylo.read(wgstree, "newick")
    tips = [c.name for c in tree.get_terminals()]
    # Find tips to prune (not in fasta)
    tips_to_prune = [term for term in tips if term not in seq_names]
    
    # Prune tips
    if(len(tips_to_prune) == len(tips)):
        print(len(tips), len(seq_names), 0)
        exit(1)
    else:
        for tip in tips_to_prune:
            tree.prune(tip)    
        #remove zero-length branches
        nozerotree = remove_zero_length_branches(tree) 

        # Write pruned tree
        print(len(tips), len(seq_names), len(tree.get_terminals()))

        Phylo.write(nozerotree, pruned_tree_file, "newick")


def filter_seqs_to_tree(fastaf, treefile, output_fasta):
    """
    Filter sequences in a FASTA file to only those present as tips in a tree file.
    Args:
        fastaf (str): Path to input FASTA file.
        treefile (str): Path to tree file (Newick format).
        output_fasta (str): Path to output filtered FASTA file.
    """
    # Get tip names from tree
    tree = Phylo.read(treefile, "newick")
    tip_names = set([term.name for term in tree.get_terminals()])
    # Parse FASTA and filter
    records = [rec for rec in SeqIO.parse(fastaf, "fasta")]
    nrecords = len(records)
    goodrecords = [rec for rec in records if rec.id in tip_names]
    # Write filtered FASTA
    with open(output_fasta, "w") as out:
        SeqIO.write(goodrecords, out, "fasta")

    print(nrecords, len(tip_names), len(goodrecords))


In [17]:
# Run FUBAR using HyPhy on the codon alignment and IQ-TREE treefile
def run_fubar(codontrimf, treefile, fubarout):
    errfile = fubarout + ".err"    
    outfile = fubarout + ".out"    
    with open(fubarout, "w") as errfile, open(outfile, "w") as outfile:
        subprocess.run([
            "hyphy", "fubar",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--output", fubarout
        ], check=True,stderr=errfile, stdout=outfile)


def run_absrel(codontrimf, treefile, absrelout, branches=None):
    """Run aBSREL using HyPhy on the codon alignment and IQ-TREE treefile."""
    if branches is None:
        command = [
            "hyphy", "absrel",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--output", absrelout
        ]
        print(" ".join(command))
        subprocess.run(command, check=True)
    else:
        command = [
            "hyphy", "absrel",
            "--alignment", codontrimf,
            "--tree", treefile,
            "--branches", branches,
            "--output", absrelout]
        print(" ".join(command))
        subprocess.run(command, check=True)

In [12]:
#annotate tree with clade assignments

def annotate_tree(treefile,cladesfile):
    tree = Phylo.read(treefile, "newick")

    
    # Find a specific clade (e.g., the common ancestor of C and D)
    clade_to_highlight = tree.common_ancestor("C", "D")

    # Set the color and width of the branch leading to this clade
    clade_to_highlight.color = "red"
    clade_to_highlight.width = 3

    # Draw the tree (e.g., to ASCII or using Matplotlib)
    Phylo.draw_ascii(tree)

In [56]:
#mafft  --auto --keeplength --add inputs/RSVA_all_aligned.fasta  refs/RSVA.fasta > inputs/RSVA_all_refaligned.fasta
#mafft  --auto --keeplength --add inputs/RSVB_all_aligned.fasta  refs/RSVB.fasta > inputs/RSVB_all_refaligned.fasta


target="RSVB"
reff = f'refs/{target}.fasta'
gfff = f'refs/{target}.gff3'
fastaf = f'inputs/{target}_all_aligned.fasta'  # <-nextstrain aligned output 
wgstree = f'{target}_tree.nwk'


extract_reference_gene_aa(reff, gfff, f'./refgenes/{target}_')

db = gffutils.create_db(gfff, dbfn=':memory:')
geneids = [cds.attributes['gene'][0] for cds in db.features_of_type('CDS', order_by='start')]




In [57]:
#genelist = ["G", "F", "L","M"]
#genelist = ['NS1', 'NS2', 'N', 'P', 'SH', 'M2-1', 'M2-2']
genelist = geneids
for gene in genelist:

    reffasta = f"refgenes/{target}_{gene}_aa.fasta"                #reference amino acid sequence

    #input sequences
    gfasta = f"geneseqs/{target}_{gene}_sequences.fasta"           #nucleotide sequences
    gaafasta = f"geneseqs/{target}_{gene}_aa.fasta"                #amino acid sequences

    #make codon alignment
    gmafftf = f"geneseqs/{target}_{gene}_aa_mafft.fasta"            #amino acid alignment
    codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

    #filter tree / sequences to match
    wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

    #hyphy outputs
    fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output


    #extract gene region for all samples, filter sequences with > 20% Ns
    extract_gene_sequences(fastaf, gfff, gene, gfasta, gaafasta)
    nkeep, ndrop = filter_n_sequences(gfasta, gaafasta, gfasta, gaafasta, n_threshold=0.05)
    print(f"{gene:5s} kept {nkeep:4d}, dropped{ndrop:4d}  (pass:{nkeep / (nkeep + ndrop):6.2%})")


    #get codon alignment, trim to last stop codon
    get_codon_alignment(gaafasta, gfasta, reffasta, gmafftf, codonf)
    trim_stop_codons(codonf, codonf, mincodons=20)


NS1   kept 1788, dropped 100  (pass:94.70%)
mafft --keeplength --auto --op 3.0 --anysymbol --add geneseqs/RSVB_NS1_aa.fasta refgenes/RSVB_NS1_aa.fasta


CalledProcessError: Command '['mafft', '--keeplength', '--auto', '--op', '3.0', '--anysymbol', '--add', 'geneseqs/RSVB_NS1_aa.fasta', 'refgenes/RSVB_NS1_aa.fasta']' returned non-zero exit status 1.

In [ ]:
genelist = ['NS1', 'NS2', 'N', 'P', 'SH', 'M2-1', 'M2-2']
genelist = geneids
#run fubar for all genes in genelist
for gene in genelist:
    #codon alignment
    codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

    #filter tree / sequences to match
    wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

    #hyphy outputs
    fubarout = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output

    #prune tree to sequences, filter sequences to tree
    prune_tree_to_seqs(wgstree, codonf, wgstrim)
    filter_seqs_to_tree(codonf, wgstree, codonf)

    run_fubar(codonf, wgstrim, fubarout)

Removing 0 zero-length branches
1936 1692 1692
1692 1936 1692
Removing 0 zero-length branches
1936 1713 1682
1713 1936 1682
Removing 0 zero-length branches
1936 1433 1404
1433 1936 1404
Removing 0 zero-length branches
1936 1802 1767
1802 1936 1767
Removing 0 zero-length branches
1936 1809 1778
1809 1936 1778
Removing 0 zero-length branches
1936 1667 1634
1667 1936 1634
Removing 0 zero-length branches
1936 1951 1916
1951 1936 1916


In [ ]:
#run absrel for all genes in genelist
for gene in genelist:
    #codon alignment
    codonf = f"geneseqs/{target}_{gene}_codonaln.fasta"             #nucleotide codon alignment

    #filter tree / sequences to match
    wgstrim = f"geneseqs/{target}_{gene}_tree_pruned.nwk"           #pruned wg tree

    #hyphy outputs
    absrelout = f"hyphy/{target}_{gene}_absrel.json"                  #FUBAR output

    #prune tree to sequences, filter sequences to tree
    #prune_tree_to_seqs(wgstree, codonf, wgstrim)
    #filter_seqs_to_tree(codonf, wgstree, codonf)

    #run_absrel(codonf, wgstrim, absrelout)



Removing 0 zero-length branches
1936 1724 1692
1724 1936 1692


NameError: name 'fubarout' is not defined

In [46]:
import pandas as pd
import json

def parse_fubar_table(json_file):
    """
    Reads a fubar output file and parses out the table as a pandas DataFrame.
    Args:
        json_file (str): Path to the JSON file.
    Returns:
        pd.DataFrame: Parsed table as DataFrame.
    """
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    jsontable = data["MLE"]["content"]["0"]
    df = pd.DataFrame(jsontable)
    headers = [h[0] for h in data["MLE"]["headers"]]
    #df.columns = headers
    if(len(df.columns) > len(headers)):
        headers[len(headers):len(df.columns)] = ["x" + str(i) for i in range(len(headers), len(df.columns))]
    df.columns = headers
    df.reset_index(names='codon', inplace=True)
    df['dataset'] = json_file.replace(".json","").split("/")[-1]
    return df


In [47]:
genelist = geneids

for gene in genelist:
    fubarjson = f"hyphy/{target}_{gene}_fubar.json"                  #FUBAR output
    fubercsv = f"hyphy/{target}_{gene}_fubar_table.csv"                  #FUBAR table output
    parse_fubar_table(fubarjson).to_csv(fubercsv, index=False)